# Tinh chỉnh bộ phân loại liên quan nhạc lý (TÙY CHỌN, G6)

**Notebook này CHƯA được chạy** (không có Colab/GPU trong phiên làm việc). Nó là kế hoạch có sẵn mã để nhóm thử: huấn luyện đầu phân loại trên vector e5 *đóng băng* với siêu tham số quét kỹ hơn, và xuất ra đúng định dạng `e5-logreg.json` mà `backend/app/admin/relevance/classifier.py` đọc được.

Không đụng LLM. Adapter/reranker (tinh chỉnh bộ mã hóa) cần thêm mã suy luận ở backend nên **chưa tích hợp**; xem `docs/MODEL_CARD.md` mục Hạn chế.

Chạy: Colab (CPU là đủ) → Runtime → Run all, sau khi tải `data/relevance/handwritten.jsonl` và `eval/cases.jsonl` từ repo lên.

In [ ]:
!pip -q install scikit-learn sentence-transformers numpy
import json, numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

rows = [json.loads(l) for l in open('handwritten.jsonl', encoding='utf-8') if l.strip()]
texts = ['passage: ' + r['text'] for r in rows]   # e5 cần tiền tố passage:
y = np.array([r['label'] for r in rows])
groups = [r['group'] for r in rows]
enc = SentenceTransformer('intfloat/multilingual-e5-small')
X = enc.encode(texts, normalize_embeddings=True)
print(X.shape, y.mean())

In [ ]:
# Quét C bằng kiểm chéo theo nhóm chủ đề (không để câu cùng nhóm nằm ở cả train lẫn test)
best = None
for C in [0.5, 1, 2, 4, 8, 16, 32]:
    p = cross_val_predict(LogisticRegression(C=C, class_weight='balanced', max_iter=3000), X, y, cv=GroupKFold(5), groups=groups, method='predict_proba')[:, 1]
    auc = roc_auc_score(y, p); print(C, round(auc, 4))
    if best is None or auc > best[0]: best = (auc, C)
print('tốt nhất', best)

In [ ]:
# Xuất cùng định dạng với app/ml/relevance_train.export_models 
lr = LogisticRegression(C=best[1], class_weight='balanced', max_iter=3000, random_state=0).fit(X, y)
spec = {'type': 'embedding_logreg', 'embedder': 'multilingual-e5-small', 'task': 'RETRIEVAL_DOCUMENT', 'dim': int(X.shape[1]), 'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0])}
json.dump(spec, open('e5-logreg.json', 'w'), separators=(',', ':'))
# Sau đó: chép vào backend/models/relevance/, chạy lại `python -m app.ml.relevance_train` để cập nhật manifest.json (SHA-256), rồi pytest.